# Unión: train (tienda 44, 2016-2017) + oil + items + holidays_events

Parte del CSV ya filtrado (`train_tienda44_2016_2017.csv`, generado en el notebook anterior) y le une, por izquierda (*left join*, para no perder ninguna fila de `train`):

1. `oil.csv` por `date`
2. `items.csv` por `item_nbr`
3. `holidays_events.csv` por `date` (deduplicado primero, porque algunas fechas tienen más de una fila)

**Antes de correr:** asegúrate de haber corrido antes `train_tienda44_2016_2017.ipynb` (para que exista el archivo `train_tienda44_2016_2017.csv` en `C:/Tesis`).

In [ ]:
import pandas as pd
import os

BASE_DIR = "C:/Tesis"


## 1. Cargar train_filtrado (tienda 44, 2016-2017)

In [ ]:
train_filtrado = pd.read_csv(os.path.join(BASE_DIR, "train_tienda44_2016_2017.csv"))
train_filtrado["date"] = pd.to_datetime(train_filtrado["date"])

print("Filas:", len(train_filtrado))
print("Columnas:", list(train_filtrado.columns))


## 2. Unir con oil.csv (por date)
`oil.csv` tiene una sola fila por fecha, así que este join no agrega ni quita filas.

In [ ]:
oil = pd.read_csv(os.path.join(BASE_DIR, "oil.csv"))
oil["date"] = pd.to_datetime(oil["date"])

df = train_filtrado.merge(oil, on="date", how="left")

print("Filas:", len(df), "(debería ser igual a train_filtrado)")
print("Columnas:", list(df.columns))


## 3. Unir con items.csv (por item_nbr)
`items.csv` tiene una sola fila por producto, tampoco agrega ni quita filas.

In [ ]:
items = pd.read_csv(os.path.join(BASE_DIR, "items.csv"))

df = df.merge(items, on="item_nbr", how="left")

print("Filas:", len(df), "(debería seguir igual)")
print("Columnas:", list(df.columns))


## 4. Unir con holidays_events.csv (por date)
Acá primero hay que deduplicar: algunas fechas tienen más de una fila (ej. feriado nacional + local el mismo día). Nos quedamos con la primera fila por fecha, para no duplicar filas de `train` al unir. También renombramos `type` a `holiday_type` para que no se confunda con otras columnas `type` (por ejemplo si más adelante agregas `stores.csv`).

In [ ]:
holidays = pd.read_csv(os.path.join(BASE_DIR, "holidays_events.csv"))
holidays["date"] = pd.to_datetime(holidays["date"])

print("Filas antes de deduplicar:", len(holidays))
holidays = holidays.drop_duplicates(subset="date", keep="first")
print("Filas después de deduplicar:", len(holidays))

holidays = holidays.rename(columns={"type": "holiday_type"})


In [ ]:
df = df.merge(holidays, on="date", how="left")

print("Filas:", len(df), "(debería seguir igual a train_filtrado)")
print("Columnas:", list(df.columns))


## 5. Revisión de la base final
Los `NaN` en `dcoilwtico` (fines de semana/feriados sin precio) y en las columnas de `holidays_events` (días que no son feriado) son esperables, no un error.

In [ ]:
print(f"Filas: {len(df):,}")
print(f"Columnas: {df.shape[1]}")
print()
df.info()


In [ ]:
df.isna().sum()


In [ ]:
df.head(10)


## 6. Guardar el resultado

In [ ]:
OUTPUT_PATH = os.path.join(BASE_DIR, "train_tienda44_union.csv")
df.to_csv(OUTPUT_PATH, index=False)
print("Guardado en:", OUTPUT_PATH)
